# Cleaning, standardising, and linking the five workforce datasets

## Purpose

This notebook shows how to create clean analysis tables from the five raw sources without editing the original Excel files:

1. Astea Utilisation Data
2. Employee Pay Report (Active and Inactive sheets are one source)
3. Financial Demands
4. Technician Type Costing
5. Price Book Data

The process is designed to be auditable: keep raw values, create standardised columns alongside them, validate each key, then merge only at the appropriate level of detail.

**Do not use `Astea_Utilisation_Cleaned_draft.xlsx` in this notebook.**

## How the datasets link

```text
Employee Pay ── Person Number / Employee_ID ── Astea Utilisation ── Demand ID + Person Number ── Financial Demands
                                                        │
                                                        ├── SA Type ID ── Technician Type Costing
                                                        │
                                                        └── Customer ID + Activity Code + Demand Type* ── Price Book Data

*Demand Type comes from Financial Demands after the validated demand-person link.
```

- `Person Number` / `Employee_ID`: employee-level link. Standardise both to `person_id` as **text**.
- `Demand ID` + `Person Number`: safest activity-to-financial link because a demand can involve multiple people.
- `SA Type ID`: links each utilisation activity to a standard technician-costing row.
- `Customer ID` + `Demand_Type` + `Activity Code`: a price-book **candidate key**. Use effective dates before selecting a specific price.

In [14]:
%pip install pyxlsb

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

# Mount Google Drive in Colab.
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# This folder contains both the notebook and all raw Excel files.
DATA_FOLDER = Path("/content/drive/MyDrive/MIS399")

# Confirm the main source file is available.
assert (DATA_FOLDER / "Astea_Utilisation_Data.xlsx").exists(), (
    f"File not found in: {DATA_FOLDER}"
)

# Create an output folder inside the same MIS399 Google Drive folder.
OUTPUT_FOLDER = DATA_FOLDER / "cleaned_standardised_outputs"
OUTPUT_FOLDER.mkdir(exist_ok=True)

print("Reading Excel files from:", DATA_FOLDER)
print("Saving outputs to:", OUTPUT_FOLDER)

# Required for .xlsb files.
try:
    import pyxlsb
except ImportError:
    print("Run this once, then run this setup cell again:")
    print("%pip install pyxlsb")

Mounted at /content/drive
Reading Excel files from: /content/drive/MyDrive/MIS399
Saving outputs to: /content/drive/MyDrive/MIS399/cleaned_standardised_outputs


## 1. Load raw data and preserve it

Never overwrite `*_raw`. All cleaning is performed on copies. This makes it possible to trace a cleaned value back to the original source.

In [16]:
# Load each raw Excel dataset into a separate DataFrame.
# Keep these *_raw tables unchanged so all later cleaning steps remain traceable.

# Astea: activity-level workforce utilisation records
util_raw = pd.read_excel(DATA_FOLDER / "Astea_Utilisation_Data.xlsx")

# Employee Pay Report: current active employee snapshot
active_raw = pd.read_excel(
    DATA_FOLDER / "Employee_Pay_Report.xlsx",
    sheet_name="Active Employee"
)

# Employee Pay Report: inactive/former employee salary and termination history
inactive_raw = pd.read_excel(
    DATA_FOLDER / "Employee_Pay_Report.xlsx",
    sheet_name="Inactive Employee"
)

# Financial Demands: demand-level reported cost, price, and duration information
financial_raw = pd.read_excel(DATA_FOLDER / "Financial_Demands.xlsx")

# Technician Type Costing: standard cost/price rate cards by technician type.
# engine="pyxlsb" is required because this file uses Excel's .xlsb format.
costing_raw = pd.read_excel(
    DATA_FOLDER / "Technician_Type_Costing.xlsb",
    engine="pyxlsb"
)

# Price Book Data: customer, activity, demand-type, and price-book information.
# This is also an .xlsb file, so it requires pyxlsb.
price_book_raw = pd.read_excel(
    DATA_FOLDER / "Price_Book_Data.xlsb",
    engine="pyxlsb"
)

# Create a quick inventory to confirm each file loaded successfully.
# Rows show the volume of records; columns show the number of available fields.
source_inventory = pd.DataFrame({
    "source": [
        "Astea Utilisation",
        "Employee Pay - Active",
        "Employee Pay - Inactive",
        "Financial Demands",
        "Technician Type Costing",
        "Price Book",
    ],
    "rows": [
        len(util_raw),
        len(active_raw),
        len(inactive_raw),
        len(financial_raw),
        len(costing_raw),
        len(price_book_raw),
    ],
    "columns": [
        util_raw.shape[1],
        active_raw.shape[1],
        inactive_raw.shape[1],
        financial_raw.shape[1],
        costing_raw.shape[1],
        price_book_raw.shape[1],
    ],
})

# Display the source inventory table.
display(source_inventory)

,source,rows,columns
0,Astea Utilisation,434447,32
1,Employee Pay - Active,584,21
2,Employee Pay - Inactive,1181,11
3,Financial Demands,479626,37
4,Technician Type Costing,73,14
5,Price Book,52823,18


## 2. Reusable standardisation functions

Identifiers must be text so leading zeros, blank values, and different Excel number formats do not break joins. Dates and numeric fields are standardised separately.

In [17]:
# Remove leading/trailing spaces from text values.
# Missing values remain missing rather than becoming the text "nan".
def clean_text(series):
    """Trim text while retaining missing values."""
    return series.astype("string").str.strip()


# Standardise identifier fields such as Person Number, Demand ID, and Order ID.
# IDs are stored as text to preserve leading zeros and prevent numeric-format issues.
def clean_id(series):
    """Create a text identifier; remove Excel's trailing .0 from numeric IDs."""

    # First remove unnecessary spaces.
    result = clean_text(series)

    # Remove ".0" added when Excel stores an identifier as a number.
    # Example: "400157.0" becomes "400157".
    result = result.str.replace(r"\.0$", "", regex=True)

    # Convert blank and placeholder values into real missing values.
    return result.replace({
        "": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
        "None": pd.NA
    })


# Standardise reporting categories such as Trade, Department, or Activity Code.
# Missing values are labelled clearly, but are not guessed or inferred.
def clean_category(series, unknown_label="Unknown"):
    """Create an analysis category without guessing a missing value."""

    return (
        clean_text(series)
        .replace({
            "": pd.NA,
            "NULL": pd.NA,
            "null": pd.NA
        })
        .fillna(unknown_label)
    )


# Convert a column to numeric values.
# Invalid text values become missing values (NaN) instead of causing an error.
def clean_number(series):
    return pd.to_numeric(series, errors="coerce")


# Create a quick data-quality profile for a potential merge key.
# This helps check whether the key is complete and whether duplicates may create
# double-counting when tables are merged.
def key_profile(df, key_columns, name):
    return pd.DataFrame([{
        "table": name,  # Name used to identify the table in the output.

        # Shows the key being checked, for example: "demand_id + person_id".
        "key": " + ".join(key_columns),

        # Total rows in the table.
        "rows": len(df),

        # Rows where every column required for the key is present.
        "rows_with_complete_key": df[key_columns].notna().all(axis=1).sum(),

        # Number of rows involved in duplicated keys.
        # Duplicates may be valid, but must be understood before merging.
        "duplicate_key_rows": df.duplicated(key_columns, keep=False).sum()
    }])

## 3. Clean Astea Utilisation

This is the central activity table. The output remains at **activity row** level. `Trade_clean` keeps missing/`NULL` values as `Unknown`; it does not infer Trade from technician type, job title, or activity code.

In [18]:
# Create a working copy of the raw Astea data.
# The original util_raw DataFrame remains unchanged for auditability.
util = util_raw.copy()


# Standardise identifiers used to link Astea data with other datasets.
# All IDs are converted to cleaned text values to avoid Excel formatting issues.
util["person_id"] = clean_id(util["Person Number"])
util["demand_id"] = clean_id(util["Demand ID"])
util["order_id"] = clean_id(util["Order ID"])
util["customer_id"] = clean_id(util["Customer ID"])
util["sa_type_id"] = clean_id(util["SA Type ID"])


# Standardise descriptive fields used for analysis and reporting.
util["activity_code"] = clean_text(util["Activity Code"])

# Convert Date to a valid datetime field.
# Invalid or unreadable dates become missing values rather than causing an error.
util["work_date"] = pd.to_datetime(util["Date"], errors="coerce")

# Create clean reporting categories.
# Missing Trade remains "Unknown"; do not infer Trade from other fields.
util["trade_clean"] = clean_category(util["Trade"])

# Use a specific label so missing departments are clear in charts and summaries.
util["department_clean"] = clean_category(
    util["Department"],
    "Unknown department"
)

# Preserve missing Billable? values explicitly until business definitions are confirmed.
util["billable_code"] = clean_category(
    util["Billable?"],
    "Missing"
)


# List all time-related fields that should be numeric.
# These include productive work, overtime, leave, private activities, and lost time.
hour_fields = [
    "reg_duration",
    "over_duration_1",
    "over_duration_2",
    "holiday_duration",
    "travel_duration",
    "Approved Leave",
    "Approved OT",
    "Private Activities",
    "Lost Time"
]

# Convert each duration field to numeric.
# Missing or invalid values are set to zero for time aggregation.
for field in hour_fields:
    util[field] = clean_number(util[field]).fillna(0)


# Create analysis-ready time measures.

# Standard regular work hours.
util["regular_hours"] = util["reg_duration"]

# Total overtime combines first- and second-level overtime.
util["overtime_hours"] = (
    util["over_duration_1"] +
    util["over_duration_2"]
)

# Total recorded work hours includes regular, overtime, holiday, and travel time.
# It deliberately excludes approved leave, private activities, and lost time.
util["recorded_hours"] = (
    util["regular_hours"] +
    util["overtime_hours"] +
    util["holiday_duration"] +
    util["travel_duration"]
)

# Create a month field for monthly trends and Power BI reporting.
util["month"] = util["work_date"].dt.to_period("M").astype("string")


# Create a data-quality summary for important fields.
util_quality = pd.DataFrame({
    "check": [
        "Missing person ID",
        "Missing demand ID",
        "Unknown trade",
        "Missing date",
        "Non-positive recorded hours"
    ],
    "rows": [
        util["person_id"].isna().sum(),
        util["demand_id"].isna().sum(),
        (util["trade_clean"] == "Unknown").sum(),
        util["work_date"].isna().sum(),
        (util["recorded_hours"] <= 0).sum()
    ]
})

# Calculate the percentage of all Astea rows affected by each quality check.
util_quality["pct_of_rows"] = 100 * util_quality["rows"] / len(util)

# Display the data-quality results.
display(util_quality)

,check,rows,pct_of_rows
0,Missing person ID,0,0.000000
1,Missing demand ID,0,0.000000
2,Unknown trade,3265,0.751530
3,Missing date,0,0.000000
4,Non-positive recorded hours,120104,27.645259


In [19]:
zero_hour_rows = util[util["recorded_hours"] <= 0].copy()

zero_hour_summary = (
    zero_hour_rows.groupby("Activity Code", dropna=False)
    .agg(
        rows=("Demand ID", "size"),
        approved_leave=("Approved Leave", "sum"),
        private_activities=("Private Activities", "sum"),
        lost_time=("Lost Time", "sum")
    )
    .sort_values("rows", ascending=False)
)

display(zero_hour_summary.head(20))

,rows,approved_leave,private_activities,lost_time
Activity Code,,,,
NC_LOSTTIME,60527,0.00,0.00,4842.16
NC_Annual_Leave,15973,129558.19,0.00,0.00
PUBLIC_HOLIDAY,10299,82444.11,0.00,0.00
NC_Sick_Leave,5756,43312.85,0.00,0.00
AOT,4999,0.00,0.00,0.00
NC_Office,4299,0.00,25169.54,0.00
NC_RDO,4245,32502.33,0.00,0.00
NC_Training,3061,0.00,19274.33,0.00
ELEC_1STD,1515,0.00,0.00,0.00


In [20]:
print("Zero recorded hours:", (util["recorded_hours"] == 0).sum())
print("Negative recorded hours:", (util["recorded_hours"] < 0).sum())

Zero recorded hours: 120104
Negative recorded hours: 0


## 4. Clean Employee Pay Report

The active and inactive sheets have different structures, so retain them as separate tables. Do not compare salary values until pay basis is aligned: the active sheet identifies Hourly/Salaried, while the inactive sheet does not.

In [21]:
# Create a working copy of the active employee sheet.
# This preserves active_raw as the unchanged source data.
active = active_raw.copy()


# Standardise the employee identifier for linking to Astea utilisation data.
active["person_id"] = clean_id(active["Person Number"])

# Convert the enterprise hire date into a usable date field.
# Invalid dates become missing values.
active["hire_date"] = pd.to_datetime(
    active["Enterprise Hire Date"],
    errors="coerce"
)

# Standardise current pay fields.
# Base Salary/Hourly Rate may be an hourly rate or salary value depending on pay basis.
active["base_pay_rate"] = clean_number(
    active["Base Salary/Hourly Rate"]
)

# Annualized Salary provides an annual-pay measure where available.
active["annualised_salary"] = clean_number(
    active["Annualized Salary"]
)

# Identify whether the employee is hourly or salaried.
# This must be considered before comparing pay values.
active["pay_basis"] = clean_category(
    active["Hourly/Salaried"],
    "Unknown pay basis"
)

# Standardise the department field for reporting.
active["department_clean"] = clean_category(
    active["Department Name"],
    "Unknown department"
)


# Create a working copy of the inactive employee sheet.
# This contains former employee and salary-history information.
inactive = inactive_raw.copy()


# Standardise Employee_ID so it can link to person_id in the other datasets.
inactive["person_id"] = clean_id(inactive["Employee_ID"])

# Convert termination and salary-change fields to valid dates.
inactive["termination_date"] = pd.to_datetime(
    inactive["Termination Date"],
    errors="coerce"
)

inactive["salary_change_date"] = pd.to_datetime(
    inactive["Salary Change Date"],
    errors="coerce"
)

# Convert the reported salary amount into a numeric field.
# Caution: the inactive source does not include a pay-basis field,
# so this value may represent either an hourly or annual amount.
inactive["reported_salary_amount"] = clean_number(
    inactive["Salary Amount"]
)

# Standardise the department field for reporting.
inactive["department_clean"] = clean_category(
    inactive["Department Name"],
    "Unknown department"
)


# Create one final record per former employee for leaver analysis.
# Sorting ensures the latest termination/salary-history record is retained.
leaver_latest = (
    inactive
    .sort_values([
        "person_id",
        "termination_date",
        "salary_change_date"
    ])
    .drop_duplicates("person_id", keep="last")
    .copy()
)


# Check whether active employee IDs are complete and unique.
display(key_profile(
    active,
    ["person_id"],
    "active_pay"
))

# Check the inactive salary-history grain.
# Multiple records per person are expected because salary may change over time.
display(key_profile(
    inactive,
    ["person_id", "salary_change_date"],
    "inactive_pay_history"
))

,table,key,rows,rows_with_complete_key,duplicate_key_rows
0,active_pay,person_id,584,584,0


,table,key,rows,rows_with_complete_key,duplicate_key_rows
0,inactive_pay_history,person_id + salary_change_date,1181,1181,0


## 5. Clean Financial Demands

This table is at a financial demand-person line level. Keep cost and price as reported fields. Validate their business definitions with Finance before calling any difference “margin”.

In [22]:
# Create a working copy of the Financial Demands dataset.
# The original financial_raw DataFrame remains unchanged.
financial = financial_raw.copy()


# Standardise identifiers used to connect Financial Demands
# with Astea utilisation and employee data.
financial["person_id"] = clean_id(financial["Person Number"])
financial["demand_id"] = clean_id(financial["Demand ID"])
financial["order_id"] = clean_id(financial["Order ID"])


# Standardise Demand Type for reporting and price-book matching.
# Missing values are labelled rather than guessed.
financial["demand_type"] = clean_category(
    financial["Demand_Type"],
    "Unknown demand type"
)


# Convert the scheduled date to a valid datetime field.
# Invalid values become missing rather than causing errors.
financial["scheduled_date"] = pd.to_datetime(
    financial["Scheduled - Date"],
    errors="coerce"
)


# Convert reported financial fields to numeric values.
# Any invalid values become missing (NaN).
financial["total_cost"] = clean_number(financial["Total Cost"])
financial["total_price"] = clean_number(financial["Total Price"])
financial["time_duration_charged"] = clean_number(
    financial["Time Duration Charged"]
)


# Calculate the difference between reported price and reported cost.
# Do not label this as confirmed margin until Finance validates:
# - the meaning of Total Cost and Total Price;
# - whether overheads are included;
# - whether rows are at a consistent demand/person level.
financial["reported_price_less_cost"] = (
    financial["total_price"] -
    financial["total_cost"]
)


# Profile the business merge key.
# Order ID + Demand ID + Person ID is required because the same demand/person
# combination can occur under different orders.
display(key_profile(
    financial,
    ["order_id", "demand_id", "person_id"],
    "financial_demands"
))

,table,key,rows,rows_with_complete_key,duplicate_key_rows
0,financial_demands,order_id + demand_id + person_id,479626,479626,0


## 6. Clean Technician Type Costing and Price Book Data

These are reference tables. Standardising their keys first ensures they can be safely used in lookups and coverage checks.

In [39]:
costing = costing_raw.copy()
costing["sa_type_id"] = clean_id(costing["SA Type ID"])
costing["sa_type_clean"] = clean_category(costing["SA Type"], "Unknown service-agent type")
for field in ["Regular Cost", "1st Overtime Cost (%)", "2nd Overtime Cost (%)", "Holiday Cost (%)", "Travel Time Cost", "Regular Price", "1st Overtime Price (%)", "2nd Overtime Price (%)", "Holiday Price (%)"]:
    if field in costing.columns:
        costing[field] = clean_number(costing[field])
costing = costing.drop_duplicates("sa_type_id")

price_book = price_book_raw.copy()
# Clean column names first.
price_book.columns = price_book.columns.astype(str).str.strip()

# Standardise the Price Book record key.
price_book["pbookh_id_clean"] = clean_id(price_book["pbookh_id"])
price_book["pbookl_id_clean"] = clean_id(price_book["pbookl_id"])
price_book["customer_id"] = clean_id(price_book["Customer ID"])

# Standardise fields used to find a candidate pricing rule.
price_book["demand_type"] = clean_category(
    price_book["Demand_Type"],
    "Unknown demand type"
)
price_book["activity_code"] = clean_text(
    price_book["Activity Code"]
)

# Use the exact header names in your workbook.
date_fields = [
    "Date (From) - Price Book Line",
    "Date (To) - Price Book Line",
    "Date (From) - Price Book Header",
    "Date (To) - Price Book Header"
]

for field in date_fields:
    price_book[field] = pd.to_datetime(
        price_book[field],
        errors="coerce"
    )

# Convert price-related fields to numbers.
for field in ["Actual Price", "Unit Cost", "Base Price", "Quantity"]:
    price_book[field] = clean_number(price_book[field])

# Check whether the composite Price Book record key is unique.
display(key_profile(
    price_book,
    ["pbookh_id_clean", "pbookl_id_clean", "customer_id"],
    "price_book"
))

,table,key,rows,rows_with_complete_key,duplicate_key_rows
0,price_book,pbookh_id_clean + pbookl_id_clean + customer_id,52823,47488,124


In [40]:
# ---------------------------------------------------------
# Remove exact duplicate Price Book rows
# ---------------------------------------------------------
# This removes rows only when every column value is identical.
# It does NOT remove valid rows with the same IDs but different
# effective dates, prices, or other attributes.

price_book_before = len(price_book)

price_book = (
    price_book
    .drop_duplicates()
)

price_book_after = len(price_book)

print("Rows before removing exact duplicates:", price_book_before)
print("Exact duplicate rows removed:", price_book_before - price_book_after)
print("Rows after removing exact duplicates:", price_book_after)

Rows before removing exact duplicates: 52823
Exact duplicate rows removed: 62
Rows after removing exact duplicates: 52761


## 7. Validate relationships before merging

A match rate must be reviewed before interpreting joined data. The Financial Demands business key is **Order ID + Demand ID + Person ID**.

A high match rate does not prove fields have the same business meaning; it only proves that the identifiers align. Keep unmatched keys for investigation. The Price Book result is a candidate-key check only because the operational sources do not contain Price Book Header/Line IDs.


In [41]:
# =========================================================
# Relationship validation: show match coverage before merging
# =========================================================

# A. Utilisation → Financial Demands
# Full business key: Order ID + Demand ID + Person ID.
financial_keys = financial[["order_id", "demand_id", "person_id"]].drop_duplicates()
util_financial_check = util[["order_id", "demand_id", "person_id"]].merge(
    financial_keys.assign(financial_match=True),
    on=["order_id", "demand_id", "person_id"],
    how="left"
)

# B. Utilisation → Technician Type Costing
# Reference lookup key: SA Type ID.
util_cost_check = util[["sa_type_id"]].merge(
    costing[["sa_type_id"]].assign(costing_match=True),
    on="sa_type_id",
    how="left"
)

# C. Activity/Financial → Price Book candidate key
# Demand Type is supplied by Financial Demands after the full financial key is matched.
util_financial_bridge = util[["order_id", "demand_id", "person_id", "customer_id", "activity_code"]].merge(
    financial[["order_id", "demand_id", "person_id", "demand_type"]].drop_duplicates(),
    on=["order_id", "demand_id", "person_id"],
    how="left"
)
price_keys = price_book[["customer_id", "demand_type", "activity_code"]].drop_duplicates()
price_check = util_financial_bridge.merge(
    price_keys.assign(price_book_candidate_match=True),
    on=["customer_id", "demand_type", "activity_code"],
    how="left"
)

relationship_quality = pd.DataFrame({
    "relationship": [
        "Utilisation → Financial (Order + Demand + Person)",
        "Utilisation → Technician Costing (SA Type)",
        "Activity/Financial → Price Book candidate key"
    ],
    "left_rows": [len(util_financial_check), len(util_cost_check), len(price_check)],
    "matched_rows": [
        util_financial_check.financial_match.notna().sum(),
        util_cost_check.costing_match.notna().sum(),
        price_check.price_book_candidate_match.notna().sum()
    ]
})
relationship_quality["match_pct"] = 100 * relationship_quality.matched_rows / relationship_quality.left_rows
display(relationship_quality)

# Create a unique list of Astea keys that do not have a Financial Demands match.
unmatched_financial_keys = (
    util_financial_check[util_financial_check["financial_match"].isna()]
    [["order_id", "demand_id", "person_id"]]
    .drop_duplicates()
    .sort_values(["order_id", "demand_id", "person_id"])
)
print("Unique unmatched Order + Demand + Person keys:", len(unmatched_financial_keys))
display(unmatched_financial_keys.head(20))


,relationship,left_rows,matched_rows,match_pct
0,Utilisation → Financial (Order + Demand + Person),434447,315060,72.519778
1,Utilisation → Technician Costing (SA Type),434447,434447,100.000000
2,Activity/Financial → Price Book candidate key,434447,266019,61.231635


Unique unmatched Order + Demand + Person keys: 119387


,order_id,demand_id,person_id
391573,SV2405030538@@1,70645679,316
24049,SV2405070340@@1,78056274,13000882
288632,SV2405070340@@1,80237307,236
220854,SV2405070340@@1,83993369,412328
220859,SV2405070340@@1,84044463,412328
220860,SV2405070340@@1,84094943,412328
220861,SV2405070340@@1,84159001,412328
220862,SV2405070340@@1,84220837,412328
288758,SV2405070340@@1,85008100,236
220994,SV2405070340@@2,88651437,412328


In [42]:
# =========================================================
# Create and export the master merged audit file
# =========================================================
# GRAIN OF THIS FILE: one original Astea utilisation activity row.
#
# The file is for checking relationships and row-level analysis. Financial
# values may repeat across multiple activity rows for the same key, so do NOT
# sum financial_total_cost or financial_total_price across this master file
# without first aggregating back to Order + Demand + Person grain.
#
# Price Book fields are NOT joined as an exact price because several valid
# effective-dated Price Book lines can share the same business attributes.
# Instead, a price_book_candidate_match flag is included.

# 1. Summarise Financial Demands to one row per full business key.
financial_summary = (
    financial
    .groupby(["order_id", "demand_id", "person_id"], as_index=False, dropna=False)
    .agg(
        financial_line_count=("demand_id", "size"),
        demand_type=("demand_type", "first"),
        financial_total_cost=("total_cost", "sum"),
        financial_total_price=("total_price", "sum"),
        financial_charged_hours=("time_duration_charged", "sum"),
        financial_price_less_cost=("reported_price_less_cost", "sum")
    )
)

# 2. Technician Type Costing is a one-row-per-SA-Type lookup.
costing_lookup = costing[[
    "sa_type_id", "sa_type_clean", "Regular Cost", "Regular Price", "Travel Time Cost"
]].drop_duplicates("sa_type_id")

# 3. Active and former employee lookups are kept separate, then joined by person_id.
active_lookup = active[[
    "person_id", "hire_date", "pay_basis", "base_pay_rate", "annualised_salary"
]].drop_duplicates("person_id")
leaver_lookup = leaver_latest[[
    "person_id", "termination_date", "reported_salary_amount"
]].drop_duplicates("person_id")

# 4. Create one candidate Price Book flag per full operational/financial key.
price_candidate_lookup = (
    price_check
    .groupby(["order_id", "demand_id", "person_id"], as_index=False, dropna=False)
    .agg(price_book_candidate_match=("price_book_candidate_match", "max"))
)

# 5. Merge validated one-row lookup tables onto each original Astea activity row.
# validate="m:1" stops the process if a lookup unexpectedly has duplicate keys.
master_merged = (
    util
    .merge(financial_summary, on=["order_id", "demand_id", "person_id"], how="left", validate="m:1")
    .merge(costing_lookup, on="sa_type_id", how="left", validate="m:1")
    .merge(active_lookup, on="person_id", how="left", validate="m:1")
    .merge(leaver_lookup, on="person_id", how="left", validate="m:1")
    .merge(price_candidate_lookup, on=["order_id", "demand_id", "person_id"], how="left", validate="m:1")
)

# 6. Add transparent flags for checking coverage in Excel or Power BI.
master_merged["financial_match"] = master_merged["financial_line_count"].notna()
master_merged["costing_match"] = master_merged["sa_type_clean"].notna()
master_merged["price_book_candidate_match"] = master_merged["price_book_candidate_match"].fillna(False)
master_merged["employment_status_from_pay_file"] = np.select(
    [master_merged["pay_basis"].notna(), master_merged["termination_date"].notna()],
    ["Active", "Former"],
    default="Not found in pay extract"
)

# 7. Export the master file and key quality checks to Google Drive.
master_file = OUTPUT_FOLDER / "MIS399_master_merged_audit.csv"
quality_file = OUTPUT_FOLDER / "relationship_quality.csv"
unmatched_file = OUTPUT_FOLDER / "unmatched_utilisation_to_financial.csv"

master_merged.to_csv(master_file, index=False)
relationship_quality.to_csv(quality_file, index=False)
unmatched_financial_keys.to_csv(unmatched_file, index=False)

print("Master merged file saved to Google Drive:")
print(master_file)
print("Rows in master file:", len(master_merged))
print("Other checking files saved:")
print(quality_file)
print(unmatched_file)

# In Colab, this displays a clickable Drive-file link inside the notebook.
if IN_COLAB:
    display(FileLink(master_file, result_html_prefix="Open/download master CSV: "))

# Review a small sample before using the full file.
display(master_merged.head())


/tmp/ipykernel_885/3159771770.py:63: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_merged["price_book_candidate_match"] = master_merged["price_book_candidate_match"].fillna(False)


Master merged file saved to Google Drive:
/content/drive/MyDrive/MIS399/cleaned_standardised_outputs/MIS399_master_merged_audit.csv
Rows in master file: 434447
Other checking files saved:
/content/drive/MyDrive/MIS399/cleaned_standardised_outputs/relationship_quality.csv
/content/drive/MyDrive/MIS399/cleaned_standardised_outputs/unmatched_utilisation_to_financial.csv


/content/drive/MyDrive/MIS399/cleaned_standardised_outputs/MIS399_master_merged_audit.csv

,Action Group ID,Department,Cost Centre,Security Role ID,Role,SA Type ID,SA Type,Trade,Access - Desktop,Access - Mobile,Demand ID,Order ID,Activity Code,Site,Customer ID,RequestType,Billable?,Date,Start,End,Pay Cost Centre - Demand,Approved Leave,Approved OT,Private Activities,Lost Time,reg_duration,over_duration_1,over_duration_2,holiday_duration,travel_duration,Person Number,Supervisor Number,person_id,demand_id,order_id,customer_id,sa_type_id,activity_code,work_date,trade_clean,department_clean,billable_code,regular_hours,overtime_hours,recorded_hours,month,financial_line_count,demand_type,financial_total_cost,financial_total_price,financial_charged_hours,financial_price_less_cost,sa_type_clean,Regular Cost,Regular Price,Travel Time Cost,hire_date,pay_basis,base_pay_rate,annualised_salary,termination_date,reported_salary_amount,price_book_candidate_match,financial_match,costing_match,employment_status_from_pay_file
0,TERMINATED,DENNSWAOAU,CL00165980,TeamLeader,Team Leader,NSW_PLUMB,NSW Plumbing,PLUMB,N,N,52962770,SV2404120402@@1,PLUM_1STD,PR00105800,DENNSWAOAU,RM,N,2025-02-18,07:00:00,16:07:00,CL00165980,0.0,0.0,0.0,0.0,9.00,0.00,0.12,0.0,0.0,13000861,354,13000861,52962770,SV2404120402@@1,DENNSWAOAU,NSW_PLUMB,PLUM_1STD,2025-02-18,PLUMB,DENNSWAOAU,N,9.00,0.12,9.12,2025-02,1.0,Labor,809.78,0.0,9.50,-809.78,NSW Plumbing,88.5,91.6,88.5,NaT,<NA>,NaN,NaN,2025-08-25,53.80965,True,True,True,Former
1,TERMINATED,DENNSWAOAU,CL00165980,TeamLeader,Team Leader,NSW_PLUMB,NSW Plumbing,PLUMB,N,N,57519360,SV2407100210@@1,PLUM_1STD,PRSP00022,BGISPCLOAU,RM,N,2025-07-16,07:00:00,08:00:00,CL00165980,0.0,0.0,0.0,0.0,1.00,0.00,0.00,0.0,0.0,13000861,354,13000861,57519360,SV2407100210@@1,BGISPCLOAU,NSW_PLUMB,PLUM_1STD,2025-07-16,PLUMB,DENNSWAOAU,N,1.00,0.00,1.00,2025-07,1.0,Labor,88.50,0.0,1.00,-88.50,NSW Plumbing,88.5,91.6,88.5,NaT,<NA>,NaN,NaN,2025-08-25,53.80965,True,True,True,Former
2,TERMINATED,DENNSWAOAU,CL00165980,TeamLeader,Team Leader,NSW_PLUMB,NSW Plumbing,PLUMB,N,N,58686791,SV2407310311@@1,PLUM_1STD,PR00107241,DENNSWAOAU,RM,N,2025-01-23,00:00:00,00:05:00,CL00165980,0.0,0.0,0.0,0.0,0.00,0.00,0.08,0.0,0.0,13000861,354,13000861,58686791,SV2407310311@@1,DENNSWAOAU,NSW_PLUMB,PLUM_1STD,2025-01-23,PLUMB,DENNSWAOAU,N,0.00,0.08,0.08,2025-01,1.0,Labor,8.85,0.0,0.08,-8.85,NSW Plumbing,88.5,91.6,88.5,NaT,<NA>,NaN,NaN,2025-08-25,53.80965,True,True,True,Former
3,TERMINATED,DENNSWAOAU,CL00165980,TeamLeader,Team Leader,NSW_PLUMB,NSW Plumbing,PLUMB,N,N,59862592,SV2408190067@@1,PLUM_1STD,PRSP00015,BGISPCLOAU,RM,N,2025-02-27,07:00:00,15:00:00,CL00165980,0.0,0.0,0.0,0.0,8.00,0.00,0.00,0.0,0.0,13000861,354,13000861,59862592,SV2408190067@@1,BGISPCLOAU,NSW_PLUMB,PLUM_1STD,2025-02-27,PLUMB,DENNSWAOAU,N,8.00,0.00,8.00,2025-02,1.0,Labor,708.00,0.0,8.00,-708.00,NSW Plumbing,88.5,91.6,88.5,NaT,<NA>,NaN,NaN,2025-08-25,53.80965,True,True,True,Former
4,TERMINATED,DENNSWAOAU,CL00165980,TeamLeader,Team Leader,NSW_PLUMB,NSW Plumbing,PLUMB,N,N,60024169,SV2408190306@@1,PLUM_1STD,PR00107830,DENNSWAOAU,RM,N,2025-03-07,09:50:00,16:02:00,CL00165980,0.0,0.0,0.0,0.0,6.17,0.03,0.00,0.0,0.0,13000861,354,13000861,60024169,SV2408190306@@1,DENNSWAOAU,NSW_PLUMB,PLUM_1STD,2025-03-07,PLUMB,DENNSWAOAU,N,6.17,0.03,6.20,2025-03,1.0,Labor,549.36,0.0,6.50,-549.36,NSW Plumbing,88.5,91.6,88.5,NaT,<NA>,NaN,NaN,2025-08-25,53.80965,True,True,True,Former


## 8. Using the master file safely

The master file is saved in Google Drive at `cleaned_standardised_outputs/MIS399_master_merged_audit.csv`. It is an **Astea activity-row-level** audit file, intended for checking matches and doing row-level exploration.

### Before analysing totals

- Sum `recorded_hours` only at the desired reporting grain.
- Financial amounts are summarised at Order + Demand + Person level but can repeat across several activity rows. Aggregate to that same three-part key before summing `financial_total_cost`, `financial_total_price`, or `financial_price_less_cost`.
- `price_book_candidate_match = True` means a possible Price Book combination exists. It does not mean a specific price was selected. Use Price Book Header/Line IDs and effective dates to choose an actual price.
- `Unknown` Trade is a source-data quality category. Do not replace it with a guessed trade.

### Files written to Google Drive

| File | Purpose |
|---|---|
| `MIS399_master_merged_audit.csv` | Main activity-level checking file. |
| `relationship_quality.csv` | Match percentages across all relationships. |
| `unmatched_utilisation_to_financial.csv` | Unique Order + Demand + Person keys in Astea that have no Financial Demands match. |
